In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.stats import ttest_ind
import seaborn as sns
from math import sqrt
import re
from matplotlib.ticker import ScalarFormatter
import matplotlib.ticker as mticker

cell_types = ["Exc", "Pvalb", "Sst", "Vip", "Htr3a"]
colors = [
    "tab:red",
    "tab:blue",
    "tab:green",
    "tab:purple",
    "tab:purple",
]
layer_colors = [
    "tab:red",
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:purple",
]
layers = ["L1", "L2/3", "L4", "L5", "L6"]

In [ ]:
def str2list(str_time):
    str_time = str_time[1:-1]
    array_list = str_time.strip(",").split(", ")
    return np.stack([float(ii) for ii in array_list])

In [ ]:
def plot_func(axes, power_layers_sil, conditions_list, colors, ii):
    for condition in conditions_list:
        mean_power = power_layers_sil.loc[power_layers_sil["condition"].isin([condition]), "mean"]
        se_power = power_layers_sil.loc[power_layers_sil["condition"].isin([condition]), "se"]

        if mean_power.empty:
            continue
        else:
            data = np.vstack(np.asanyarray(mean_power))
            data_se = np.vstack(np.asanyarray(se_power))

        if condition != 'no perturb':
            c = colors[ii]
            ii += 1
        else:
            c = 'k'

        for ax, i in zip(axes[1:], range(1, 6)):
            ax.plot(power_layers_sil["f"][0], data[i, :], label=condition, color=c)
            ax.fill_between(
                power_layers_sil["f"][0][1:],
                data[i, 1:] - 1.96 * data_se[i, 1:], # 95% confidence interval
                data[i, 1:] + 1.96 * data_se[i, 1:], 
                facecolor=c,
                alpha=0.3,
            )
            ax.set_xlim(0,100)
            if condition == conditions_list[0]:  # Only add text for the first condition to avoid repetition
                ax.text(
                    1,
                    1.25,
                    layers[i],
                    horizontalalignment="right",
                    verticalalignment="top",
                    fontsize=8,
                    alpha=1,
                    transform=ax.transAxes,
                    fontweight="bold",
                ) 
            # ax.text(
            #     1,
            #     0.95,
            #     layers[i],
            #     horizontalalignment="right",
            #     verticalalignment="top",
            #     fontsize=8,
            #     alpha=1,
            #     transform=ax.transAxes,
            #     fontweight="bold",
            # ) 
            sns.despine()  # Removes top and right spines by default
            # Position the left spine outward by 10 points
            ax.spines["left"].set_position(("outward", 3))
            # Position the bottom spine outward by 10 points
            ax.spines["bottom"].set_position(("outward", 3))

            ax.set_xlim([4, 100])

            ax.set_xticks([4, 20, 40, 60, 80, 100])
            ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
            ax.set_xticks([10, 30, 50, 70, 90], minor=True)
            ax.tick_params(axis='x', which='major', length=5)
            ax.tick_params(axis='x', which='minor', length=3)

            # Format y-axis with scientific notation for large values
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_powerlimits(
                (-3, 3)
            )  # Use scientific notation for values with >5 digits
            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
            ax.tick_params(axis='y', labelsize=8)

    return ii

## Fig 4A: Layer by layer silencing

In [ ]:
net_name = "core"  # change to "core_<variant>" for a model variant
power_layers_sil = pd.read_csv(f"{net_name}/metrics/csv_files_dg_memory_efficient_layers_perturb_-2000_all/dg_all_dirs_mua_power_layer.csv")

In [ ]:
power_layers_sil["power"] = power_layers_sil["power"].apply(str2list)
power_layers_sil["f"] = power_layers_sil["f"].apply(str2list)

power_layers_sil_df = power_layers_sil.groupby(["condition", "layer"]).agg(
    {
        "power": list,
        "f": list,
    }
)
power_layers_sil_df.reset_index(inplace=True)
power_layers_sil_df["mean"] = power_layers_sil_df["power"].apply(np.mean, axis=0)
power_layers_sil_df["std"] = power_layers_sil_df["power"].apply(np.std, axis=0)
power_layers_sil_df["se"] = power_layers_sil_df["power"].apply(lambda x: np.std(x, axis=0) / np.sqrt(len(x)))
power_layers_sil_df["f"] = power_layers_sil_df["f"].apply(np.mean, axis=0)

In [ ]:
fig, axs = plt.subplots(
    ncols=1, nrows=5, sharex=True, figsize=(2.5, 5.5), dpi=300
)
fig.patch.set_facecolor("none")
f = power_layers_sil_df.loc[0, "f"]

colors = np.delete(plt.colormaps["plasma"](np.linspace(0, 1, 7)), 6, axis=0)

axes = axs
ii = 1
ii = plot_func(axes, power_layers_sil_df, list(['no perturb', 'silencing L1', 'silencing L23',
                                     'silencing L4', 'silencing L5', 'silencing L6']), colors, ii)
legend_ax = axs[0]
legend_ax.legend(
    handles=axs[1].get_legend_handles_labels()[0],
    labels=axs[1].get_legend_handles_labels()[1],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
    ncol=2,
)
axes[0].set_title("Layer by layer\n silencing", fontsize=10)
axs[0].get_xaxis().set_visible(False)
axs[0].get_yaxis().set_visible(False)
axs[0].set_axis_off()

fig.supylabel("Power [(spks/s)²]", fontsize=10, x=0.1)
fig.supxlabel("Frequency (Hz)", fontsize=10, y=0.035)
fig.tight_layout()

fig.savefig(
    "paper_figs/fig4_dg_power_layers_silencing_Jul27_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)


In [ ]:
fig, axs = plt.subplots(
    ncols=1, nrows=5, sharex=True, figsize=(2.5, 5.5), dpi=300
)
fig.patch.set_facecolor("none")

colors = np.delete(plt.colormaps["plasma"](np.linspace(0, 1, 7)), 6, axis=0)
ii = 1

axes = axs
data_noperturb = np.vstack(np.asanyarray(power_layers_sil_df.loc[power_layers_sil_df["condition"].isin(['no perturb']), "mean"]))
data_noperturb_se = np.vstack(np.asanyarray(power_layers_sil_df.loc[power_layers_sil_df["condition"].isin(['no perturb']), "se"]))
for condition, ax, i in zip(["L2/3", "L4", "L5", "L6"], axes[1:], range(1, 5)):
    data = np.vstack(np.asanyarray(power_layers_sil_df.loc[power_layers_sil_df["condition"].isin([condition]), "mean"]))
    data_se = np.vstack(np.asanyarray(power_layers_sil_df.loc[power_layers_sil_df["condition"].isin([condition]), "se"]))

    ax.plot(power_layers_sil_df["f"][0], data[i, :], label=condition, color=colors[ii])
    ax.fill_between(
        power_layers_sil_df["f"][0][1:],
        data[i, 1:] - 1.96 * data_se[i, 1:],
        data[i, 1:] + 1.96 * data_se[i, 1:],
        facecolor=colors[ii],
        alpha=0.3,
    )
    ax.plot(power_layers_sil_df["f"][0], data_noperturb[i, :], label='no perturb', color='k')
    ax.fill_between(
        power_layers_sil_df["f"][0][1:],
        data_noperturb[i, 1:] - 1.96 * data_noperturb_se[i, 1:],
        data_noperturb[i, 1:] + 1.96 * data_noperturb_se[i, 1:],
        facecolor='k',
        alpha=0.3,
    )
    ax.set_xlim(0, 100)
    ax.text(
        1, 0.95, layers[i],
        horizontalalignment="right", verticalalignment="top",
        fontsize=8, alpha=1, transform=ax.transAxes, fontweight="bold",
    )
    sns.despine()
    ax.spines["left"].set_position(("outward", 3))
    ax.spines["bottom"].set_position(("outward", 3))
    ax.set_xlim([4, 100])
    ax.set_xticks([4, 20, 40, 60, 80, 100])
    ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
    ax.set_xticks([10, 30, 50, 70, 90], minor=True)
    ax.tick_params(axis='x', which='major', length=5)
    ax.tick_params(axis='x', which='minor', length=3)
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-3, 3))
    ax.yaxis.set_major_formatter(formatter)
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
    ax.tick_params(axis='y', labelsize=8)
    # ii += 1

legend_ax = axs[0]
legend_ax.legend(
    handles=axs[1].get_legend_handles_labels()[0],
    labels=["isolated layer", "no perturb"],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
)
axes[0].set_title("Isolated layers", fontsize=10)
axs[0].get_xaxis().set_visible(False)
axs[0].get_yaxis().set_visible(False)
axs[0].set_axis_off()

fig.supylabel("Power [(spks/s)²]", fontsize=10, x=0.1)
fig.supxlabel("Frequency (Hz)", fontsize=10, y=0.035)
fig.tight_layout()

fig.savefig(
    "paper_figs/fig4_dg_power_isolated_layers_Jul27_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)


## Suppl Fig 4A: Layer silencing, all layers

In [ ]:
fig, axes = plt.subplots(
    ncols=1, nrows=5, sharex=True, figsize=(2.5, 5.5), dpi=300
)  # , sharey="col"
fig.patch.set_facecolor("none")
f = power_layers_sil_df.loc[0, "f"]
ii = 0
# colors = np.delete(plt.colormaps["Accent_r"](np.linspace(0, 1, 8)), 4, axis=0)
colors = np.delete(plt.colormaps["plasma"](np.linspace(0, 1, 7)), 6, axis=0) # 
ii = plot_func(axes, power_layers_sil_df, list(['no perturb', 'silencing L23 & L5', 'silencing L23 & L6', 
                                     'silencing L5 & L6', 'silencing L23, L5 & L6']), colors, ii)
# Place legend in the row above (index ii-1, since L2/3 is at ii=0 for layers[1:])
legend_ax = axes[0]  # First row for this column
legend_ax.legend(
    handles=axes[1].get_legend_handles_labels()[0],
    labels=axes[1].get_legend_handles_labels()[1],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
)
axes[0].set_title("Silencing of layer\n combinations", fontsize=10)
# axes[-1].set_xlabel("Frequency (Hz)")
# Hide first row axis
axes[0].get_xaxis().set_visible(False)
axes[0].get_yaxis().set_visible(False)
axes[0].set_axis_off()

fig.supylabel("Power [(spks/s)²]", fontsize=10, x=0.1)
fig.supxlabel("Frequency (Hz)", fontsize=10, y=0.035)
fig.tight_layout()

fig.savefig(
    "paper_figs/fig4_dg_power_layers_silencing_suppl_Jul27_26.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

## Fig 4B: L4 inh population silencing

In [ ]:
power_L4_inh_pop_sil = pd.read_csv(f"{net_name}/metrics/csv_files_dg_memory_efficient_L4only_silence_inh/dg_all_dirs_mua_power_layer_L4only_silence_inh.csv")

In [ ]:
power_L4_inh_pop_sil["power"] = power_L4_inh_pop_sil["power"].apply(str2list)
power_L4_inh_pop_sil["f"] = power_L4_inh_pop_sil["f"].apply(str2list)

power_L4_inh_pop_sil_df = power_L4_inh_pop_sil.groupby(["condition", "layer"]).agg(
    {
        "power": list,
        "f": list,
    }
)
power_L4_inh_pop_sil_df.reset_index(inplace=True)
power_L4_inh_pop_sil_df["mean"] = power_L4_inh_pop_sil_df["power"].apply(np.mean, axis=0)
power_L4_inh_pop_sil_df["std"] = power_L4_inh_pop_sil_df["power"].apply(np.std, axis=0)
power_L4_inh_pop_sil_df["se"] = power_L4_inh_pop_sil_df["power"].apply(lambda x: np.std(x, axis=0) / np.sqrt(len(x)))
power_L4_inh_pop_sil_df["f"] = power_L4_inh_pop_sil_df["f"].apply(np.mean, axis=0)

In [ ]:
def plot_func2(axes, power_df, conditions_list, colors, ii):
    for condition in conditions_list:
        mean_power = power_df.loc[power_df["condition"].isin([condition]), "mean"]
        se_power = power_df.loc[power_df["condition"].isin([condition]), "se"]

        mean_power_array = np.vstack(np.asanyarray(mean_power))
        se_power_array = np.vstack(np.asanyarray(se_power))

        if condition != 'L4':
            c = colors[ii]
            ii += 1
        else:
            c = 'k'
        
        # Clean up label by removing "silencing " from the beginning
        if condition.startswith("silencing "):
            label = condition.replace("silencing ", "") if condition.startswith("silencing ") else condition

            label_perc = 1 if label.startswith("all") else label.split(" ")[0]
            label_pop = label.split(" ")[-1]

            # Replace i23/i4/i5/i6 prefix with readable layer names
            layer_name_map = {'i23': 'L2/3 ', 'i4': 'L4 ', 'i5': 'L5 ', 'i6': 'L6 '}
            for key, val in layer_name_map.items():
                if label_pop.startswith(key):
                    label_pop = val + label_pop[len(key):]
                    break

            label = "{}% {}".format(round(float(label_perc) * 100, 2), label_pop)
        else:
            label = condition

        if label == 'L4':
            label = 'L4 - no perturb'
            
        # for axes, i in zip(axes, range(2, 3)):
        axes.plot(power_df["f"][0], mean_power_array[2, :], label=label, color=c)
        axes.fill_between(
            power_df["f"][0],
            mean_power_array[2, :] - se_power_array[2, :],
            mean_power_array[2, :] + se_power_array[2, :],
            color=c,
            alpha=0.3,
        )
        sns.despine(ax=axes)  # Removes top and right spines by default
        # Position the left spine outward by 10 points
        axes.spines["left"].set_position(("outward", 3))
        # Position the bottom spine outward by 10 points
        axes.spines["bottom"].set_position(("outward", 3))

        axes.set_xlim([4, 100])

        axes.set_xticks([4, 20, 40, 60, 80, 100])
        axes.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
        axes.set_xticks([10, 30, 50, 70, 90], minor=True)
        axes.tick_params(axis='x', which='major', length=5)
        axes.tick_params(axis='x', which='minor', length=3)

        # Format y-axis with scientific notation for large values
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits(
            (-3, 3)
        )  # Use scientific notation for values with >5 digits
        axes.yaxis.set_major_formatter(formatter)
        axes.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
        axes.tick_params(axis='y', labelsize=8)

    return ii


In [ ]:
fig, axs_array = plt.subplots(3, 2, figsize=(3.8, 5.5), dpi=300, sharex=True)

axs = axs_array.flatten()

# --- Pvalb: Main plot (all traces) with inset zoom-in on partial silencing ---
jj = 0
axes = axs[jj]
colors_pvalb = plt.colormaps["Blues"](np.linspace(0, 1, 10))[2:]
ii = 0

# Main plot: L4 baseline + all silencing conditions
ii = plot_func2(
    axes,
    power_L4_inh_pop_sil_df,
    list(
        [
            "L4",
            "silencing all i4Pvalb",
            "silencing 0.8333 i4Pvalb",
            "silencing 0.75 i4Pvalb",
            "silencing 0.625 i4Pvalb",
            "silencing 0.5 i4Pvalb",
            "silencing 0.25 i4Pvalb",
            "silencing 0.1 i4Pvalb",
        ]
    ),
    colors_pvalb,
    ii,
)
axes.set_title("Pvalb", fontsize=10)

# Inset: zoom-in on the lower amplitude partial silencing traces
axins = axes.inset_axes([0.5, 0.4, 0.5, 0.45])
ii_ins = 1  # skip index 0 used for "silencing all i4Pvalb"
ii_ins = plot_func2(
    axins,
    power_L4_inh_pop_sil_df,
    list(
        [
            "L4",
            "silencing 0.8333 i4Pvalb",
            "silencing 0.75 i4Pvalb",
            "silencing 0.625 i4Pvalb",
            "silencing 0.5 i4Pvalb",
            "silencing 0.25 i4Pvalb",
            "silencing 0.1 i4Pvalb",
        ]
    ),
    colors_pvalb,
    ii_ins,
)
axins.set_xlim([4, 100])
axins.set_xticks([4, 40, 100])
axins.tick_params(labelsize=5)
formatter_ins = ScalarFormatter(useMathText=True)
formatter_ins.set_powerlimits((-3, 3))
axins.yaxis.set_major_formatter(formatter_ins)
axins.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
# Match offset text (x10^n) font size to tick labels
axins.yaxis.get_offset_text().set_fontsize(5)
axs[jj].indicate_inset_zoom(axins, edgecolor="gray")

# Combined legend: L4-no perturb first, then ascending silencing % (10% → 100%)
jj += 1
main_handles, main_labels = axes.get_legend_handles_labels()
reorder_idx = [0] + list(range(len(main_handles) - 1, 0, -1))
axs[jj].legend(
    handles=[main_handles[i] for i in reorder_idx],
    labels=[main_labels[i] for i in reorder_idx],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
)
axs[jj].get_xaxis().set_visible(False)
axs[jj].get_yaxis().set_visible(False)
axs[jj].set_axis_off()

# --- Sst ---
jj += 1
ii = 0
colors = plt.colormaps["Greens"](np.linspace(0, 1, 10))[2:]
axes = axs[jj]
ii = plot_func2(
    axes,
    power_L4_inh_pop_sil_df,
    list(
        [
            "L4",
            "silencing all i4Sst",
            "silencing 0.8333 i4Sst",
            "silencing 0.75 i4Sst",
            "silencing 0.625 i4Sst",
            "silencing 0.5 i4Sst",
            "silencing 0.25 i4Sst",
            "silencing 0.1 i4Sst",
        ]
    ),
    colors,
    ii,
)
axes.set_title("Sst", fontsize=10)
jj += 1
sst_handles, sst_labels = axs[jj - 1].get_legend_handles_labels()
reorder_idx = [0] + list(range(len(sst_handles) - 1, 0, -1))
axs[jj].legend(
    handles=[sst_handles[i] for i in reorder_idx],
    labels=[sst_labels[i] for i in reorder_idx],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
)
axs[jj].get_xaxis().set_visible(False)
axs[jj].get_yaxis().set_visible(False)
axs[jj].set_axis_off()

# --- Vip ---
jj += 1
ii = 0
colors = plt.colormaps["Purples"](np.linspace(0, 1, 10))[2:]
axes = axs[jj]
ii = plot_func2(
    axes,
    power_L4_inh_pop_sil_df,
    list(
        [
            "L4",
            "silencing all i4Vip",
            "silencing 0.8333 i4Vip",
            "silencing 0.75 i4Vip",
            "silencing 0.625 i4Vip",
            "silencing 0.5 i4Vip",
            "silencing 0.25 i4Vip",
            "silencing 0.1 i4Vip",
        ]
    ),
    colors,
    ii,
)
axes.set_title("Vip", fontsize=10)
jj += 1
vip_handles, vip_labels = axs[jj - 1].get_legend_handles_labels()
reorder_idx = [0] + list(range(len(vip_handles) - 1, 0, -1))
axs[jj].legend(
    handles=[vip_handles[i] for i in reorder_idx],
    labels=[vip_labels[i] for i in reorder_idx],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    mode=None,
    borderaxespad=0.0,
    fontsize=6,
)
axs[jj].get_xaxis().set_visible(False)
axs[jj].get_yaxis().set_visible(False)
axs[jj].set_axis_off()

# Restore full box for inset and reset all spines to default position
for spine in axins.spines.values():
    spine.set_visible(True)
    spine.set_position(("outward", 0))

fig.supylabel("Power [(spks/s)²]", fontsize=10, x=0.05)
fig.supxlabel("Frequency (Hz)", fontsize=10, y=0.035, x=0.35)
fig.suptitle("Isolated L4 simulations: gradual\nsilencing of interneuron populations", fontsize=10, y=0.98)
fig.tight_layout()
fig.savefig(
    "paper_figs/fig4_dg_L4sim_power_inh_pop_silencing_Jul27_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
